<a href="https://colab.research.google.com/github/nehamba2017-stack/ai-agents/blob/main/DocumentQ%26A_RAG_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Necessary Libraries

In [ ]:
  !pip install faiss-cpu sentence-transformers langchain openai PyPDF2 tiktoken langchain-community pypdf

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.document_loaders import DirectoryLoader
from google.colab import files
import os

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create data folder
os.makedirs('data', exist_ok=True)

print("Environment setup complete.")

### Upload Your Document

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

### Move Uploaded Document to Data Directory

In [ ]:
import os

# Check if any files were uploaded
if not uploaded:
    print("No file was uploaded. Please run the cell above to upload a file.")
else:
    # Assuming only one file was uploaded, get the filename
    uploaded_filename = list(uploaded.keys())[0]

    # Define the destination path
    destination_path = os.path.join('data', uploaded_filename)

    # Move the file
    os.rename(uploaded_filename, destination_path)

    print(f"Moved '{uploaded_filename}' to '{destination_path}'")

### Install pypdf (if using PDF documents)

In [ ]:
!pip install pypdf

### Load and Chunk Document

In [ ]:
# Load all text documents (txt, md, pdf) from the 'data' directory
all_docs = []

for file in os.listdir('data'):
    filepath = os.path.join('data', file)
    if file.endswith('.txt') or file.endswith('.md'):
        loader = TextLoader(filepath)
    elif file.endswith('.pdf'):
        loader = PyPDFLoader(filepath)
    else:
        continue
    try:
        docs = loader.load()
        all_docs.extend(docs)
    except Exception as e:
        print(f"Error loading {filepath}: {e}")


# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(all_docs)

# Ensure 'texts' variable is populated for later use
texts = [doc.page_content for doc in splits]

print(f"Split document into {len(texts)} chunks.")

### Create Embeddings and Build Vector Store

In [ ]:
    # Embed text
    embeddings = model.encode(texts)

    # Build FAISS index
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings).astype('float32'))

    print(f"Created FAISS index with {index.ntotal} embeddings.")

### Check Number of Embeddings (Optional)

In [ ]:
print(f"Number of embeddings in the FAISS index: {index.ntotal}")

### Define Search Function

In [ ]:
def search_documents(query, index, texts, model, k=3):
    """
    Searches the FAISS index for documents most similar to the query.

    Args:
        query (str): The search query.
        index (faiss.IndexFlatL2): The FAISS index.
        texts (list): A list of the original text chunks.
        model (SentenceTransformer): The embedding model.
        k (int): The number of nearest neighbors to retrieve.

    Returns:
        list: A list of the top k most relevant text chunks.
    """
    # Embed the query
    query_embedding = model.encode([query]).astype('float32')

    # Perform the search
    distances, indices = index.search(query_embedding, k)

    # Retrieve the relevant text chunks
    results = [texts[i] for i in indices[0]]

    return results

### Setup Gemini API

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then pass the key to the SDK:

In [ ]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

Before you can make any API calls, you need to initialize the Generative Model. You can also set a `system_instruction` here.

In [ ]:
# Initialize the Gemini API
# You can add a system_instruction here if desired
gemini_model = genai.GenerativeModel(
    'gemini-1.5-flash-latest',
    # system_instruction="You are a helpful assistant that answers questions based on the provided context. Be concise and directly answer the question."
)

# If you want to use a model with a system instruction, initialize it like this:
gemini_model_with_instruction = genai.GenerativeModel(
    'gemini-1.5-flash-latest',
    system_instruction="You are a helpful assistant that answers questions based on the provided context. Be concise and directly answer the question."
)

print("Gemini model initialized.")

### Define Response Generation Function (with Examples/System Instructions)

In [ ]:
def generate_rag_response(query, relevant_chunks, model, examples=None):
    """
    Generates a response using a language model based on the query and relevant document chunks.

    Args:
        query (str): The original search query.
        relevant_chunks (list): A list of relevant text chunks retrieved from the document.
        model: The language model to use for generation (e.g., a Gemini model instance).
        examples (list, optional): A list of example query-response pairs for few-shot prompting.
                                   Each example should be a dictionary with 'query' and 'answer' keys.

    Returns:
        str: The generated response.
    """
    # Combine the query and relevant chunks into a prompt
    context = "\n\n".join(relevant_chunks)

    prompt_parts = [f"Based on the following information, answer the query:\n\nContext:\n{context}\n\n"]

    if examples:
        prompt_parts.append("Examples:")
        for example in examples:
            prompt_parts.append(f"Query: {example['query']}\nAnswer: {example['answer']}\n")

    prompt_parts.append(f"Query: {query}\n\nAnswer:")

    prompt = "\n".join(prompt_parts)

    # Generate the response using the language model
    response = model.generate_content(prompt)

    return response.text

### Combine Search and Generation into One Function

In [ ]:
def chat_with_document(query, index, texts, embedding_model, llm_model, k=3, examples=None):
    """
    Combines search and generation to answer a query based on document content.

    Args:
        query (str): The user's query.
        index (faiss.IndexFlatL2): The FAISS index.
        texts (list): A list of the original text chunks.
        embedding_model (SentenceTransformer): The model used for embedding.
        llm_model: The language model used for generation (e.g., a Gemini model instance).
        k (int): The number of nearest neighbors to retrieve.
        examples (list, optional): A list of example query-response pairs for few-shot prompting.

    Returns:
        str: The generated response from the LLM.
    """
    # Search for relevant documents
    relevant_chunks = search_documents(query, index, texts, embedding_model, k=k)

    # Generate the response using the LLM
    rag_response = generate_rag_response(query, relevant_chunks, llm_model, examples=examples)

    return rag_response

### Run the RAG Example with a User Query

In [ ]:
# Example user query
user_query = "What is the purpose of this document?"

# Define example queries and answers for few-shot prompting (Optional)
examples = [
    {"query": "What is the publication date?", "answer": "The publication date is [insert date from document]."},
    {"query": "Who is the author?", "answer": "The author is [insert author name from document]."}
]

# Call the chat_with_document function
# Choose which Gemini model instance to use: gemini_model (no system instruction) or gemini_model_with_instruction (with system instruction)
# You can also uncomment the 'examples' argument to use few-shot prompting
response = chat_with_document(
    user_query,
    index,
    texts,
    model, # This is the embedding model
    gemini_model_with_instruction, # Choose your LLM model instance
    k=5, # Number of relevant chunks to retrieve
    examples=examples # Uncomment to use examples
)

# Print the generated response
print("Generated Response:")
print(response)

NameError: name 'gemini_model_with_instruction' is not defined

# Create a Widget for UI that allows a user to input a query and display the response from the `chat_with_document` function.


Import necessary libraries for creating and displaying the UI.



In [ ]:
from ipywidgets import widgets
from IPython.display import display

## Create input and output widgets



In [ ]:
query_input = widgets.Textarea(
    placeholder='Enter your query here...',
    description='Query:',
    disabled=False,
    layout=widgets.Layout(width='auto', height='100px')
)

submit_button = widgets.Button(
    description='Submit',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to submit your query',
    icon='check' # (FontAwesome icons available: https://fontawesome.com/icons?d=gallery)
)

rag_output = widgets.Output()

# Display the widgets
display(query_input, submit_button, rag_output)

Textarea(value='', description='Query:', layout=Layout(height='100px', width='auto'), placeholder='Enter your …

Button(description='Submit', icon='check', style=ButtonStyle(), tooltip='Click to submit your query')

Output()

## Define an interactive Python function that takes the input widget's value (the user's query), calls the `chat_with_document` function, and updates the output widget with the generated response.


In [ ]:
from ipywidgets import widgets, VBox, HBox
from IPython.display import display, clear_output
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader, TextLoader

# Assuming 'model' (embedding model) and 'gemini_model' (LLM) are defined in previous cells
# Ensure these are available in the global scope or passed as arguments if needed.

file_upload = widgets.FileUpload(
    accept='.txt,.md,.pdf',  # Accepted file extensions
    multiple=False,  # Allow only one file upload
    description='Upload Document',
    tooltip='Upload your text, markdown, or PDF document',
)

query_input = widgets.Textarea(
    placeholder='Enter your query here...',
    description='Query:',
    disabled=True, # Disable until document is processed
    layout=widgets.Layout(width='auto', height='100px')
)

submit_button = widgets.Button(
    description='Submit',
    disabled=True, # Disable until document is processed
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to submit your query',
    icon='check' # (FontAwesome icons available: https://fontawesome.com/icons?d=gallery)
)

rag_output = widgets.Output()

print("Upload your document:")
display(file_upload, query_input, submit_button, rag_output)

def process_document_and_query(change):
    """
    Handles file upload, processes the document, takes the user query,
    and displays the RAG response.
    Triggered by file upload.
    """
    # Clear previous output
    with rag_output:
        clear_output(wait=True)
        print("Processing document...")

    # Process the uploaded file if available
    if file_upload.value:
        uploaded_file_info = file_upload.value[0]
        uploaded_filename = uploaded_file_info['name']
        uploaded_content = uploaded_file_info['content']

        # Save the uploaded file to the 'data' directory
        destination_path = os.path.join('data', uploaded_filename)
        with open(destination_path, 'wb') as f:
            f.write(uploaded_content)

        with rag_output:
            print(f"Uploaded and saved '{uploaded_filename}' to '{destination_path}'")

        # Load and chunk the new document
        all_docs = []
        try:
            if uploaded_filename.endswith('.txt') or uploaded_filename.endswith('.md'):
                loader = TextLoader(destination_path)
            elif uploaded_filename.endswith('.pdf'):
                loader = PyPDFLoader(destination_path)
            else:
                with rag_output:
                    print("Unsupported file type.")
                # Re-enable file upload for a new attempt
                file_upload.disabled = False
                return

            docs = loader.load()
            all_docs.extend(docs)

            text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
            global splits, texts # Make splits and texts global so they can be used by chat_with_document
            splits = text_splitter.split_documents(all_docs)
            texts = [doc.page_content for doc in splits]
            with rag_output:
                print(f"Split document into {len(texts)} chunks.")

            # Create new embeddings and build a new FAISS index
            global embeddings, index # Make embeddings and index global
            embeddings = model.encode(texts)
            index = faiss.IndexFlatL2(embeddings.shape[1])
            index.add(np.array(embeddings).astype('float32'))
            with rag_output:
                print(f"Created FAISS index with {index.ntotal} embeddings.")

            # Enable query input and submit button after successful processing
            query_input.disabled = False
            submit_button.disabled = False
            file_upload.disabled = True # Disable file upload after processing one document


        except Exception as e:
            with rag_output:
                print(f"Error processing document: {e}")
            # Re-enable file upload for a new attempt
            file_upload.disabled = False
            return

# Define the function to handle query submission
def on_query_submit(b):
    """
    Processes the user query, calls the RAG function, and displays the response.
    Triggered by button click.
    """
    query = query_input.value # Get the current value of the text area

    # Clear previous output in the output area, but keep the "Processing..." message
    with rag_output:
        clear_output(wait=True)
        print("Generating response...")


    # Ensure index and texts are available (from uploaded file processing)
    if 'index' not in globals() or 'texts' not in globals():
         with rag_output:
            print("No document loaded. Please upload a document first.")
            return

    if not query:
        with rag_output:
            print("Please enter a query.")
        return


    # Call the RAG function and display the response
    with rag_output:
        try:
            # Use gemini_model or gemini_model_with_instruction based on your preference
            response = chat_with_document(
                query,
                index,
                texts,
                model, # Embedding model
                gemini_model, # LLM model (or gemini_model_with_instruction)
                k=5
            )
            clear_output(wait=True) # Clear "Generating response..."
            print("Generated Response:")
            print(response)
        except Exception as e:
            clear_output(wait=True) # Clear "Generating response..."
            print(f"An error occurred during RAG processing: {e}")


# Link the function to the submit_button's on_click event
submit_button.on_click(on_query_submit)

# Link the process_document_and_query function to the file_upload observe event
# This will trigger the function when a file is uploaded
file_upload.observe(process_document_and_query, names='value')

ModuleNotFoundError: Module langchain_community.document_loaders not found. Please install langchain-community to access this module. You can install it using `pip install -U langchain-community`